# Getting Started Notebook for MM-RegCheck

### Prerequisites
1. **Clone and install `MM-RegCheck` with the `llm` extra** (requires [`uv`](https://docs.astral.sh/uv/)):
   ```bash
   git clone https://github.com/khanhtq29vn/MM-RegCheck.git
   cd MM-RegCheck
   UV_PROJECT_ENVIRONMENT=.venv uv sync --extra llm
   ```
2. **Select the `.venv` kernel** when opening this notebook in VS Code / Jupyter (or launch JupyterLab directly with `uv run --with jupyterlab --with ipywidgets jupyter lab`).
3. **An OpenAI-compatible endpoint** that supports `logprobs=True` and `top_logprobs` — either your own API endpoint, or a free self-hosted endpoint spun up with [`local_llm_selfhost_sandbox.ipynb`](local_llm_selfhost_sandbox.ipynb) on Kaggle GPU.

## 1. Configure Endpoint & API Key

- For **thinking/reasoning models** served via vLLM/SGLang (such as `qwen3.5-9b`), set `EXTRA_BODY = {"chat_template_kwargs": {"enable_thinking": False}}` so label-token logprob elicitation (`max_tokens=1`) receives a verdict token instead of a thinking token.
- For standard non-thinking OpenAI-compatible endpoints, leave `EXTRA_BODY = {}`.

In [ ]:
import getpass
import json
import os
from pathlib import Path

BASE_URL = "https://your-endpoint.example/v1"
MODEL = "your-model-id"
API_KEY_ENV = "OPENAI_API_KEY"

if not os.environ.get(API_KEY_ENV):
    os.environ[API_KEY_ENV] = getpass.getpass(f"Enter {API_KEY_ENV}: ")

# Example for Qwen3 / Qwen3.5 on vLLM / SGLang:
# EXTRA_BODY = {"chat_template_kwargs": {"enable_thinking": False}}
EXTRA_BODY: dict = {"chat_template_kwargs": {"enable_thinking": False}}


print(f"Configured model={MODEL!r} at {BASE_URL!r} (extra_body={EXTRA_BODY})")

## 2. Quick Endpoint Sanity Check

Verify that the endpoint is reachable and returns `logprobs` on `max_tokens=1`.

In [ ]:
from openai import OpenAI

probe_client = OpenAI(base_url=BASE_URL, api_key=os.environ[API_KEY_ENV], timeout=30.0)
probe_kwargs = {"extra_body": EXTRA_BODY} if EXTRA_BODY else {}

probe = probe_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with one digit: 0, 1, or 2."}],
    max_tokens=1,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    **probe_kwargs,
)
lp = probe.choices[0].logprobs
assert lp and lp.content, "Endpoint did not return logprobs!"
print("First token:", repr(lp.content[0].token))
print("Top candidates:", [(t.token, round(t.logprob, 3)) for t in lp.content[0].top_logprobs])

## 3. Run TF-BEAR on a Single Claim

`TFBear.from_domain(...)` loads the per-domain prompt pack (`"Beauty"` or `"Healthcare"`) and wires all five stages.

In [ ]:
from mmregcheck.pillar2.llm import EndpointConfig
from mmregcheck.pillar2.tf_bear import TFBear

endpoint = EndpointConfig(base_url=BASE_URL, env_var=API_KEY_ENV, model=MODEL)

bear = TFBear.from_domain(
    domain="Beauty",
    expansion="standard",
    endpoint=endpoint,
    expansion_params={"n_paths": 3},
    extra_body=EXTRA_BODY,
    verbose=True,
)

result = bear.run(
    text_claim="phun xăm thẩm mỹ",
    retrieved_regulatory_context="['Cắt tóc', 'Gội đầu', 'Chăm sóc da']"
)

if result.diagnostics:
    print("Diagnostics:", result.diagnostics)

In [ ]:
print(json.dumps(result.to_trace_dict(), indent=2))

## 4. Experiment with Expansion Variants (`cot-sc` / `tot`)

You can switch between `"standard"`, `"cot-sc"`, and `"tot"` and tune per-expansion parameters via `expansion_params`.

In [ ]:
cot_bear = TFBear.from_domain(
    "Healthcare",
    "cot-sc",
    endpoint=endpoint,
    expansion_params={"n_paths": 3},
    extra_body=EXTRA_BODY,
)

cot_result = cot_bear.run(
    text_claim="phun xăm thẩm mỹ",
    retrieved_regulatory_context="['Cắt tóc', 'Gội đầu', 'Chăm sóc da']"
)

trace = cot_result.to_trace_dict()
print("Verdict:", trace["verdict"], "| surviving:", trace["n_surviving"], "/", trace["n_branches"])
for branch in trace["branches"]:
    print(
        f"  {branch['branch_id']}: {branch['predicted_verdict']} "
        f"(posterior={branch['posterior_prob']:.3f}, delta={branch['log_odds_delta']:+.3f}, "
        f"gate={branch['gate']})"
    )

## 5. Run on `data/dev_sample_5.jsonl`

Load the sample split included in the repository and run `bear.run_sample(sample)` (or `run_benchmark(...)` for full batch + trace output).

In [ ]:
from mmregcheck.pillar2.sample_adapter import load_samples

sample_file = Path("data/dev_sample_5.jsonl")
if not sample_file.exists():
    sample_file = Path("../data/dev_sample_5.jsonl")

samples = load_samples(sample_file)
print(f"Loaded {len(samples)} samples from {sample_file}")

first = samples[0]
sample_bear = TFBear.from_domain(
    first.domain,
    "standard",
    endpoint=endpoint,
    expansion_params={"n_paths": 1},
    extra_body=EXTRA_BODY,
)
prediction = sample_bear.run_sample(first)
print(prediction.model_dump_json(indent=2))

## 6. Equivalent CLI Usage

For full benchmark runs with incremental writes, `<predictions>.trace.jsonl`, and `--resume` support, use the CLI or batch script:

```bash
mmregcheck run-tfbear data/dev_sample_5.jsonl \
  --base-url "$TFBEAR_BASE_URL" \
  --model "$TFBEAR_MODEL" \
  --api-key-env OPENAI_API_KEY \
  --expansion standard \
  --extra-body '{"chat_template_kwargs": {"enable_thinking": false}}' \
  --out predictions.jsonl
```